In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MultipleLocator
from cmcrameri import cm

import sys
sys.path.append("../src")

from crop_model_calibration_and_run import summarize_productive_grids

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica"],
    "font.size": 11,
    "axes.linewidth": 1,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": 1,
    "ytick.major.width": 1,
})

## run

In [ ]:
grid_yield_path = Path(
    "../outputs/3.2_simulated_yield/b_maize/"
    "aquacrop_maize_grid_yield_vn_ythr1pct_left_0p6_var_260716.csv"
)

country_ensemble_yield_path = Path(
    "../outputs/3.2_simulated_yield/b_maize/"
    "aquacrop_maize_country_ensemble_yield_"
    "vn_ythr1pct_left_0p6_var_260716.csv"
)
output_plot_path = (
    Path("../../fig") / "yield_colorbar_maize_vn.svg"
)

yield_threshold = 0.01
simulation_results_df = pd.read_csv(grid_yield_path)
comparison_df = pd.read_csv(country_ensemble_yield_path)

In [ ]:
summary_all_df = summarize_productive_grids(
    simulation_results_df,
    yield_threshold=0
)

summary_threshold_df = summarize_productive_grids(
    simulation_results_df,
    yield_threshold=yield_threshold
)

summary_ratio_df = (
    summary_all_df.merge(
        summary_threshold_df,
        on="year",
        suffixes=("_all", "_threshold"),
    )
    .assign(
        grid_ratio=lambda x: x["n_gid_threshold"] / x["n_gid_all"],
    )
    .loc[:, ["year", "grid_ratio"]]
)

incomplete_ratio_df = summary_ratio_df[
    summary_ratio_df["grid_ratio"] < 1
]

print(f"No. of years with complete-loss grids: {len(incomplete_ratio_df)}")
print(
    f"Range of grid coverage: "
    f"{incomplete_ratio_df["grid_ratio"].min():.1%}~"
    f"{incomplete_ratio_df["grid_ratio"].max():.1%}"
)

incomplete_ratio_df

In [ ]:
years = comparison_df.year
y = comparison_df.yield_obs_t_ha
y_simulated = comparison_df.yield_sim_t_ha

msr = mean_squared_error(y, y_simulated)
r2 = r2_score(y, y_simulated)
print(f"RMSE: {np.sqrt(msr):.3f}")
print(f"R-squared score: {r2:.2f}")

print(f"\nMean y: {np.mean(y):.2f} t/ha")
print(f"Range of y: {y.min():.2f}~{y.max():.2f} t/ha")
print(f"Mean y_simulated: {np.mean(y_simulated):.2f} t/ha")
print(
    f"Range of y_simulated: "
    f"{y_simulated.min():.2f}~{y_simulated.max():.2f} t/ha"
)

In [ ]:
series_1 = summary_ratio_df.grid_ratio*100

cmap = cm.navia

norm = mpl.colors.Normalize(vmin=0, vmax=100)

ymin = np.floor(min(y.min(), y_simulated.min()) * 10) / 10
ymax = np.ceil(max(y.max(), y_simulated.max()) * 10) / 10

# -------------------------
# Figure with subplots
# -------------------------
fig, (ax, ax1) = plt.subplots(
    2, 1,
    figsize=(4, 1.6),
    sharex=True,
    gridspec_kw={
        "height_ratios": [3, 0.3],
        "hspace": 0.1,   # controls vertical spacing
    },
)

# -------------------------
# Main plot (yield)
# -------------------------
for gid in simulation_results_df.GID.unique():
    gid_simulation_df = simulation_results_df[
        simulation_results_df["GID"]==gid
        ]
    ax.plot(
        gid_simulation_df.year,
        gid_simulation_df.yield_sim_t_ha,
        color="#0072B2",
        lw=0.3,
        alpha=0.2,
        zorder=1
    )

ax.axhline(
    y=np.mean(y),
    color="#000000",
    linestyle="--",
    lw=1,
    alpha=0.8
)
ax.axhline(
    y=np.mean(y_simulated),
    color="#009E73",
    linestyle="--",
    lw=1,
    alpha=0.8
)

ax.plot(
    years,
    y,
    color="#000000",
    lw=1.6,
    label="FAO-detrended",
    zorder=2
)
ax.plot(
    years,
    y_simulated,
    color="#009E73",
    lw=1.6,
    label="HA-weighted ensemble",
    zorder=3
)

ax.set_ylim(ymin,ymax)
ax.set_ylabel("Yield (t/ha)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# -------------------------
# Color bar 1
# -------------------------
ax1.imshow(
    [series_1],
    aspect="auto",
    cmap=cmap,
    norm=norm,
    extent=[years.min()-0.5, years.max()+0.5, 0, 1],
)

ax1.set_yticks([])
ax1.set_ylabel("% grids", rotation=0, labelpad=20, va="center")

for spine in ax1.spines.values():
    spine.set_visible(False)

# -------------------------
# Shared x-axis formatting
# -------------------------
ax1.set_xlim(1991-0.5, 2020+0.5)

plt.show()
#plt.savefig(output_plot_path, bbox_inches="tight", dpi=300)

In [ ]:
output_directory = Path("../../fig")
data_coverage_colorbar_plot_path = (
    output_directory / "grid_coverage_colorbar.svg"
)

cmap = cm.navia

norm = mpl.colors.Normalize(vmin=0, vmax=100)

# Scalar mappable for colorbar
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

# Figure (colorbar only)
fig = plt.figure(figsize=(3, 0.6))

# [left, bottom, width, height]
cax = fig.add_axes([0.1, 0.4, 0.8, 0.2])

cbar = fig.colorbar(
    sm,
    cax=cax,
    orientation="horizontal",
)

cbar.set_label("% grids")

plt.show()
#plt.savefig(data_coverage_colorbar_plot_path, bbox_inches="tight", dpi=300)